In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import json
import os

# ======== 1. إعداد البيانات ==========

data_path = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"

# Augmentation (ممكن تضيفه بعد ما تتأكد إن الموديل بيتعلم)
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    data_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ======== 2. بناء النموذج ==========

# استيراد EfficientNetB0 بدون الطبقة النهائية
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # نجمّد أولاً

# بناء النموذج النهائي
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ======== 3. تدريب مبدئي ==========

model.fit(train_data, validation_data=val_data, epochs=5)

# ======== 4. Fine-Tuning ==========

# نفك تجميد آخر 20 طبقة فقط
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

# إعادة Compile بليرنينج ريت قليل
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                loss='categorical_crossentropy',
                metrics=['accuracy'])

# كولباكس
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2)
]

# تدريب نهائي
model.fit(train_data, validation_data=val_data, epochs=20, callbacks=callbacks)

# ======== 5. حفظ النموذج والتصنيفات ==========
model.save('flag_classifier_efficientnet_finetuned.h5')

with open('class_labels.json', 'w') as f:
    json.dump(train_data.class_indices, f)


Found 4900 images belonging to 45 classes.
Found 1213 images belonging to 45 classes.
Epoch 1/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 61s 366ms/step - accuracy: 0.0570 - loss: 3.7538 - val_accuracy: 0.0783 - val_loss: 3.6681
Epoch 2/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 55s 357ms/step - accuracy: 0.0746 - loss: 3.6997 - val_accuracy: 0.0627 - val_loss: 3.6770
Epoch 3/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 59s 384ms/step - accuracy: 0.0674 - loss: 3.6947 - val_accuracy: 0.0783 - val_loss: 3.6673
Epoch 4/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 61s 396ms/step - accuracy: 0.0748 - loss: 3.6947 - val_accuracy: 0.0783 - val_loss: 3.6580
Epoch 5/5
154/154 ━━━━━━━━━━━━━━━━━━━━ 60s 392ms/step - accuracy: 0.0850 - loss: 3.6645 - val_accuracy: 0.0783 - val_loss: 3.6609
Epoch 1/20
  1/154 ━━━━━━━━━━━━━━━━━━━━ 14:48 6s/step - accuracy: 0.0312 - loss: 3.7460

KeyboardInterrupt: 

In [5]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import os
import json

# مسار الداتا (منظمة بالشكل: data_path/class_name/images.jpg)
data_path = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"

# Augmentation بسيط + Normalization
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# تحميل بيانات التدريب
train_data = datagen.flow_from_directory(
    data_path,
    target_size=(64, 64),  # أصغر من 224x224 لتبسيط الشبكة
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

# تحميل بيانات التحقق
val_data = datagen.flow_from_directory(
    data_path,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


Found 4900 images belonging to 45 classes.
Found 1213 images belonging to 45 classes.


In [7]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_data.num_classes, activation='softmax')
])


In [8]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit(train_data, validation_data=val_data, epochs=20)


Epoch 1/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.2898 - loss: 2.7363 - val_accuracy: 0.8912 - val_loss: 0.3492
Epoch 2/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.8878 - loss: 0.3633 - val_accuracy: 0.9431 - val_loss: 0.1748
Epoch 3/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9505 - loss: 0.1488 - val_accuracy: 0.9415 - val_loss: 0.1762
Epoch 4/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.9700 - loss: 0.0938 - val_accuracy: 0.9670 - val_loss: 0.1376
Epoch 5/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.9757 - loss: 0.0782 - val_accuracy: 0.9481 - val_loss: 0.1482
Epoch 6/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.9773 - loss: 0.0680 - val_accuracy: 0.9678 - val_loss: 0.1317
Epoch 7/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - accuracy: 0.9697 - loss: 0.0801 - val_accuracy: 0.9604 - val_loss: 0.1352
Epoch 8/20
154/154 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.9903 - loss: 0.0302 - val_accu

In [9]:
model.save("cnn_flag_classifier.h5")

with open("class_labels.json", "w") as f:
    json.dump(train_data.class_indices, f)


In [12]:
from tensorflow.keras.preprocessing import image
import numpy as np
from tensorflow.keras.models import load_model
import json

# تحميل النموذج
model = load_model("cnn_flag_classifier.h5")

# تحميل أسماء التصنيفات
with open("class_labels.json") as f:
    class_indices = json.load(f)

class_labels = [k for k, v in sorted(class_indices.items(), key=lambda item: item[1])]

# تحميل صورة الاختبار
img = image.load_img(r"C:\Users\saher\Pictures\Screenshots\Screenshot 2025-07-10 220735.png", target_size=(64, 64))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# التنبؤ
pred = model.predict(img_array)
predicted_class = np.argmax(pred)
confidence = np.max(pred) * 100

print("Predicted:", class_labels[predicted_class])
print(f"Confidence: {confidence:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
Predicted: korea
Confidence: 97.99%


In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks
import json
import os
import datetime

# ============ 1. إعداد البيانات ============
data_path = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=90,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2,
    preprocessing_function=lambda x: x + 0.05 * tf.random.normal(tf.shape(x))  # نويز خفيف
)

train_data = datagen.flow_from_directory(
    data_path,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = datagen.flow_from_directory(
    data_path,
    target_size=(64, 64),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ============ 2. بناء نموذج CNN ============
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(64, 64, 3)),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D(2, 2),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(44, activation='softmax')  # عدل حسب عدد الكلاسات
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# ============ 3. إعداد الـ Callbacks ============
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
os.makedirs("models", exist_ok=True)

cb = [
    callbacks.ModelCheckpoint(
        "models/best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.CSVLogger("training_log.csv"),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

# ============ 4. التدريب ============
model.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    callbacks=cb
)

# ============ 5. حفظ النموذج النهائي والكلاسات ============
model.save("cnn_flag_classifier_augmented_45class.h5")

with open("class_labels.json", "w") as f:
    json.dump(train_data.class_indices, f)

print("✅ Model and class labels saved successfully.")


Found 5289 images belonging to 44 classes.
Found 1303 images belonging to 44 classes.
Epoch 1/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 285ms/step - accuracy: 0.0722 - loss: 3.5285
Epoch 1: val_accuracy improved from -inf to 0.36761, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 61s 363ms/step - accuracy: 0.0726 - loss: 3.5259 - val_accuracy: 0.3676 - val_loss: 2.0773
Epoch 2/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.3773 - loss: 1.9432
Epoch 2: val_accuracy improved from 0.36761 to 0.67920, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 115ms/step - accuracy: 0.3778 - loss: 1.9416 - val_accuracy: 0.6792 - val_loss: 1.0765
Epoch 3/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.6257 - loss: 1.0830
Epoch 3: val_accuracy improved from 0.67920 to 0.75595, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 108ms/step - accuracy: 0.6260 - loss: 1.0822 - val_accuracy: 0.7559 - val_loss: 0.7691
Epoch 4/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.7978 - loss: 0.6010
Epoch 4: val_accuracy improved from 0.75595 to 0.78434, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 114ms/step - accuracy: 0.7978 - loss: 0.6009 - val_accuracy: 0.7843 - val_loss: 0.7246
Epoch 5/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.8235 - loss: 0.5219
Epoch 5: val_accuracy did not improve from 0.78434
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 113ms/step - accuracy: 0.8236 - loss: 0.5217 - val_accuracy: 0.7721 - val_loss: 0.7283
Epoch 6/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.8525 - loss: 0.4176
Epoch 6: val_accuracy improved from 0.78434 to 0.83576, saving model to models/best_model.h5


...

166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 113ms/step - accuracy: 0.9571 - loss: 0.1241 - val_accuracy: 0.9348 - val_loss: 0.2849
Epoch 37/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9515 - loss: 0.1250
Epoch 37: val_accuracy did not improve from 0.93477
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 116ms/step - accuracy: 0.9515 - loss: 0.1251 - val_accuracy: 0.9325 - val_loss: 0.2701
Epoch 38/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9558 - loss: 0.1070
Epoch 38: val_accuracy improved from 0.93477 to 0.94091, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - accuracy: 0.9558 - loss: 0.1069 - val_accuracy: 0.9409 - val_loss: 0.2902
Epoch 39/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.9586 - loss: 0.1066
Epoch 39: val_accuracy did not improve from 0.94091
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 116ms/step - accuracy: 0.9586 - loss: 0.1066 - val_accuracy: 0.9332 - val_loss: 0.2557
Epoch 40/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9551 - loss: 0.1198
Epoch 40: val_accuracy improved from 0.94091 to 0.94551, saving model to models/best_model.h5


166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 113ms/step - accuracy: 0.9551 - loss: 0.1199 - val_accuracy: 0.9455 - val_loss: 0.2436
Epoch 41/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.9600 - loss: 0.0989
Epoch 41: val_accuracy did not improve from 0.94551
166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 110ms/step - accuracy: 0.9600 - loss: 0.0989 - val_accuracy: 0.9309 - val_loss: 0.2616
Epoch 42/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9592 - loss: 0.1080
Epoch 42: val_accuracy did not improve from 0.94551
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - accuracy: 0.9592 - loss: 0.1080 - val_accuracy: 0.8918 - val_loss: 0.6113
Epoch 43/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9562 - loss: 0.1259
Epoch 43: val_accuracy did not improve from 0.94551
166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - accuracy: 0.9562 - loss: 0.1258 - val_accuracy: 0.9363 - val_loss: 0.3530
Epoch 44/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 89ms/step - accuracy: 0.9652 - loss: 0.0889
Epoch 44

166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - accuracy: 0.9631 - loss: 0.1066 - val_accuracy: 0.9517 - val_loss: 0.2953
Epoch 46/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.9639 - loss: 0.0955
Epoch 46: val_accuracy did not improve from 0.95165
166/166 ━━━━━━━━━━━━━━━━━━━━ 19s 112ms/step - accuracy: 0.9638 - loss: 0.0956 - val_accuracy: 0.9156 - val_loss: 0.4150
Epoch 47/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.9569 - loss: 0.1225
Epoch 47: val_accuracy did not improve from 0.95165
166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - accuracy: 0.9569 - loss: 0.1226 - val_accuracy: 0.9025 - val_loss: 0.3980
Epoch 48/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.9620 - loss: 0.1131
Epoch 48: val_accuracy did not improve from 0.95165
166/166 ━━━━━━━━━━━━━━━━━━━━ 18s 111ms/step - accuracy: 0.9620 - loss: 0.1131 - val_accuracy: 0.9424 - val_loss: 0.2885
Epoch 49/100
166/166 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.9745 - loss: 0.0927
Epoch 49

✅ Model and class labels saved successfully.


In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import numpy as np
import json

# 1. تحميل النموذج
model = tf.keras.models.load_model("cnn_flag_classifier_augmented_45class.h5")

# 2. تحميل أسماء الكلاسات
with open("class_labels.json", "r") as f:
    class_indices = json.load(f)

inv_labels = {v: k for k, v in class_indices.items()}  # index → class_name

# 3. تحميل الصورة
img_path = r"c:\Users\saher\Pictures\Screenshots\Screenshot 2025-07-10 220735.png"
img = image.load_img(img_path, target_size=(64, 64))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# 4. التنبؤ
pred = model.predict(img_array)
pred_class = np.argmax(pred)
confidence = np.max(pred) * 100

# 5. طباعة النتيجة
label_name = inv_labels.get(int(pred_class), "Unknown")
print(f"✅ Predicted Class: {label_name}")
print(f"🎯 Confidence: {confidence:.2f}%")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
✅ Predicted Class: argentina
🎯 Confidence: 99.95%


In [6]:
import logging
from pathlib import Path
import numpy as np
import cv2
import random
import json, os, datetime
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# ============ 1. Paths & Parameters ============
DATA_DIR = Path(r"D:/github lite/cv_projects/Identifying and locating flags from aerial photography/data/flag_classification")
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

LOG_DIR = Path("logs") / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Augmentation probabilities & amounts
NOISE_FACTOR = 0.1
BLUR_PROB = 0.3
JPEG_QUALITY_RANGE = (30, 90)

# ============ 2. Logging ============
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# ============ 3. Custom Preprocessing ============
def random_blur(img):
    if random.random() < BLUR_PROB:
        k = random.choice([3,5])
        img = cv2.GaussianBlur(img, (k,k), 0)
    return img


def random_jpeg(img):
    q = random.randint(*JPEG_QUALITY_RANGE)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), q]
    _, enc = cv2.imencode('.jpg', img, encode_param)
    img = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    return img


def add_noise(img):
    noise = np.random.randn(*img.shape) * NOISE_FACTOR * 255
    img = img + noise
    return np.clip(img, 0, 255).astype(np.uint8)


def preprocess_function(img):
    # img is uint8 RGB
    img = add_noise(img)
    img = random_blur(img)
    img = random_jpeg(img)
    # Convert to float32 [0,1] for model
    return img.astype('float32') / 255.0

# ============ 4. Data Generators ============
# Note: rescaling now happens in preprocess_function (returns float32 [0,1])

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_function,
    rotation_range=90,
    zoom_range=0.2,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)
val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# Dynamically determine number of classes
NUM_CLASSES = train_gen.num_classes
logging.info(f"Detected {NUM_CLASSES} classes for classification.")

# ============ 5. Model Definition (EfficientNetB0) ============
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    input_shape=(*IMG_SIZE, 3),
    weights='imagenet'
)
base_model.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3))
# Preprocess input for EfficientNet
x = tf.keras.applications.efficientnet.preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = models.Model(inputs, outputs, name='flag_classifier')

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ============ 6. Callbacks ============
cb_list = [
    callbacks.ModelCheckpoint(
        MODEL_DIR / 'best_flag_classifier.h5',
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    callbacks.CSVLogger('training_log.csv'),
    callbacks.TensorBoard(log_dir=str(LOG_DIR), histogram_freq=1)
]

# ============ 7. Training ============
epochs = 50
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs,
    callbacks=cb_list
)

# ============ 8. Fine-tune ============
base_model.trainable = True
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=cb_list
)

# ============ 9. Save Model & Classes ============
model.save(str(MODEL_DIR / 'flag_classifier_final.h5'))
with open('class_indices.json', 'w') as f:
    json.dump(train_gen.class_indices, f)

logging.info("✅ Training complete and models saved.")


Found 5316 images belonging to 46 classes.
Found 1309 images belonging to 46 classes.


2025-07-11 04:37:12 [INFO] Detected 46 classes for classification.


Model: "flag_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 46)             │        11,822 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,390,353 (16.75 MB)

...

2025-07-11 04:38:57 [WARNING] You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


167/167 ━━━━━━━━━━━━━━━━━━━━ 105s 596ms/step - accuracy: 0.0223 - loss: 4.2184 - val_accuracy: 0.0222 - val_loss: 3.9419 - learning_rate: 0.0010
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.0228 - loss: 4.0818
Epoch 2: val_accuracy did not improve from 0.02215
167/167 ━━━━━━━━━━━━━━━━━━━━ 33s 196ms/step - accuracy: 0.0229 - loss: 4.0818 - val_accuracy: 0.0222 - val_loss: 4.0589 - learning_rate: 0.0010
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.0267 - loss: 4.0286
Epoch 3: val_accuracy did not improve from 0.02215
167/167 ━━━━━━━━━━━━━━━━━━━━ 35s 210ms/step - accuracy: 0.0268 - loss: 4.0284 - val_accuracy: 0.0222 - val_loss: 4.0923 - learning_rate: 0.0010
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.0349 - loss: 3.9714
Epoch 4: val_accuracy improved from 0.02215 to 0.04431, saving model to models\best_flag_classifier.h5


2025-07-11 04:40:48 [WARNING] You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. 


167/167 ━━━━━━━━━━━━━━━━━━━━ 44s 263ms/step - accuracy: 0.0349 - loss: 3.9714 - val_accuracy: 0.0443 - val_loss: 5.5662 - learning_rate: 0.0010
Epoch 5/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 198ms/step - accuracy: 0.0261 - loss: 3.9407
Epoch 5: val_accuracy did not improve from 0.04431
167/167 ━━━━━━━━━━━━━━━━━━━━ 43s 259ms/step - accuracy: 0.0261 - loss: 3.9406 - val_accuracy: 0.0443 - val_loss: 3.8923 - learning_rate: 0.0010
Epoch 6/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step - accuracy: 0.0332 - loss: 3.8995
Epoch 6: val_accuracy did not improve from 0.04431
167/167 ━━━━━━━━━━━━━━━━━━━━ 39s 236ms/step - accuracy: 0.0332 - loss: 3.8995 - val_accuracy: 0.0313 - val_loss: 3.8809 - learning_rate: 0.0010
Epoch 7/50
 91/167 ━━━━━━━━━━━━━━━━━━━━ 11s 150ms/step - accuracy: 0.0285 - loss: 3.8911

KeyboardInterrupt: 

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import EfficientNetB0
import json
import os
import datetime

# ============ 1. إعداد المسارات ============
DATA_DIR = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
NUM_CLASSES = 44

# ============ 2. Data Augmentation ============
def preprocess_function(img):
    # تحويل للصيغة float32 وتطبيق نويز بسيط
    img = tf.image.convert_image_dtype(img, tf.float32)
    img += 0.05 * tf.random.normal(tf.shape(img))
    return tf.clip_by_value(img, 0.0, 1.0)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_function,
    rotation_range=90,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ============ 3. بناء الموديل ============
base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False  # تجميد الطبقات الأساسية في البداية

model = models.Sequential([
    base_model,
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),    # 512 = 2^9
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),    # 256 = 2^8
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),    # 128 = 2^7
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),     # 64  = 2^6
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),     # 32  = 2^5
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============ 4. Callbacks ============
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs("models", exist_ok=True)

cb_list = [
    callbacks.ModelCheckpoint(
        "models/best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.CSVLogger("training_log.csv"),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

# ============ 5. Training ============
epochs = 50
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs,
    callbacks=cb_list
)

# ============ 6. Fine-tune ============
base_model.trainable = True
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=cb_list
)

# ============ 7. حفظ النموذج والكلاسات ============
model.save("cnn_flag_classifier_final.h5")
with open("class_labels.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("✅ Model and labels saved.")

# شغّل الكود ده، وستحصل على نموذج عميق يستخدم وحدات (neurons) بأعداد مضاعفات 2 عبر عدة طبقات، ويتعامل مع 44 فئة. إذا احتجت أي تعديل أو إضافة مثل `GaussianNoise` أو `LR scheduler` مخصص، قول لي!


Found 5316 images belonging to 46 classes.
Found 1309 images belonging to 46 classes.


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 44)             │         1,452 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,886,575 (18.64 MB)

 Trainable params: 834,444 (3.18 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

c:\Users\saher\anaconda3\envs\tf-gpu-lite\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 46), output.shape=(None, 44)

In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import EfficientNetB0
import json
import os
import datetime

# ============ 1. إعداد المسارات ============
DATA_DIR = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

# ============ 2. Data Augmentation ============
def preprocess_function(img):
    img = tf.image.convert_image_dtype(img, tf.float32)
    img += 0.05 * tf.random.normal(tf.shape(img))
    return tf.clip_by_value(img, 0.0, 1.0)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_function,
    rotation_range=90,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ======== Use dynamic number of classes ========
NUM_CLASSES = train_gen.num_classes
print(f"Detected {NUM_CLASSES} classes.")

# ============ 3. بناء الموديل ============
base_model = EfficientNetB0(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ============ 4. Callbacks ============
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs("models", exist_ok=True)

cb_list = [
    callbacks.ModelCheckpoint(
        "models/best_model.h5",
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    callbacks.CSVLogger("training_log.csv"),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

# ============ 5. Training ============
epochs = 50
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs,
    callbacks=cb_list
)

# ============ 6. Fine-tune ============
base_model.trainable = True
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=cb_list
)

# ============ 7. حفظ النموذج والكلاسات ============
model.save("cnn_flag_classifier_final.h5")
with open("class_labels.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("✅ Model and labels saved.")


Found 5316 images belonging to 46 classes.
Found 1309 images belonging to 46 classes.
Detected 46 classes.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 46)             │         1,518 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,886,641 (18.64 MB)

 Trainable params: 834,510 (3.18 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 0.0244 - loss: 3.8893
Epoch 1: val_accuracy improved from -inf to 0.04431, saving model to models/best_model.h5


167/167 ━━━━━━━━━━━━━━━━━━━━ 40s 205ms/step - accuracy: 0.0244 - loss: 3.8892 - val_accuracy: 0.0443 - val_loss: 3.8218 - learning_rate: 0.0010
Epoch 2/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.0301 - loss: 3.8346
Epoch 2: val_accuracy did not improve from 0.04431
167/167 ━━━━━━━━━━━━━━━━━━━━ 32s 193ms/step - accuracy: 0.0301 - loss: 3.8346 - val_accuracy: 0.0443 - val_loss: 3.8174 - learning_rate: 0.0010
Epoch 3/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 0.0320 - loss: 3.8209
Epoch 3: val_accuracy did not improve from 0.04431
167/167 ━━━━━━━━━━━━━━━━━━━━ 32s 194ms/step - accuracy: 0.0321 - loss: 3.8209 - val_accuracy: 0.0443 - val_loss: 3.8141 - learning_rate: 0.0010
Epoch 4/50
167/167 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 0.0355 - loss: 3.8181
Epoch 4: val_accuracy did not improve from 0.04431
167/167 ━━━━━━━━━━━━━━━━━━━━ 32s 193ms/step - accuracy: 0.0355 - loss: 3.8181 - val_accuracy: 0.0443 - val_loss: 3.8105 - learning_rate: 0.0010
Epoch 5

KeyboardInterrupt: 

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import EfficientNetV2B1
import json
import os
import datetime

# ============ 1. إعداد المسارات ============ #
DATA_DIR = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ============ 2. Data Augmentation ============ #
def preprocess_function(img):
    img = tf.image.convert_image_dtype(img, tf.float32)
    img += 0.05 * tf.random.normal(tf.shape(img))  # إضافة ضوضاء خفيفة
    return tf.clip_by_value(img, 0.0, 1.0)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_function,
    rotation_range=45,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    shear_range=0.15,
    brightness_range=(0.8, 1.2),
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print(f"✅ Detected {NUM_CLASSES} classes.")

# ============ 3. بناء الموديل ============ #
base_model = EfficientNetV2B1(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False  # Freeze مؤقت

model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.GaussianNoise(0.1),
    base_model,
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# ============ 4. Optimizer & Compile ============ #
initial_lr = 1e-2
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=initial_lr,
    first_decay_steps=1000,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-4
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-4
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============ 5. Callbacks ============ #
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs("models", exist_ok=True)

cb_list = [
    callbacks.ModelCheckpoint(
        "models/best_model.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.CSVLogger("training_log.csv"),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

# ============ 6. التدريب ============ #
epochs = 50
model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=epochs,
    callbacks=cb_list
)

# ============ 7. Fine-tuning (فتح الطبقات) ============ #
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20,
    callbacks=cb_list
)

# ============ 8. حفظ النموذج ============ #
model.save("cnn_flag_classifier_final.h5")
with open("class_labels.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("✅ Model and labels saved.")


Found 15110 images belonging to 13 classes.
Found 3772 images belonging to 13 classes.
✅ Detected 13 classes.
28456008/28456008 ━━━━━━━━━━━━━━━━━━━━ 17s 1us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gaussian_noise (GaussianNoise)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b1 (Functional)  │ (None, 1280)           │     6,931,124 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 13)             │           429 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,767,105 (29.63 MB)

 Trainable params: 833,421 (3.18 MB)

...

c:\Users\saher\anaconda3\envs\tf-gpu-lite\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 0s 609ms/step - accuracy: 0.1575 - loss: 2.4199
Epoch 1: val_accuracy improved from -inf to 0.16198, saving model to models/best_model.h5


473/473 ━━━━━━━━━━━━━━━━━━━━ 395s 814ms/step - accuracy: 0.1575 - loss: 2.4194 - val_accuracy: 0.1620 - val_loss: 2.1206 - learning_rate: 0.0054
Epoch 2/50
473/473 ━━━━━━━━━━━━━━━━━━━━ 0s 528ms/step - accuracy: 0.1672 - loss: 2.0918
Epoch 2: val_accuracy improved from 0.16198 to 0.16729, saving model to models/best_model.h5


473/473 ━━━━━━━━━━━━━━━━━━━━ 310s 655ms/step - accuracy: 0.1672 - loss: 2.0918 - val_accuracy: 0.1673 - val_loss: 2.0945 - learning_rate: 7.2770e-05
Epoch 3/50
254/473 ━━━━━━━━━━━━━━━━━━━━ 1:47 492ms/step - accuracy: 0.1646 - loss: 2.0945

KeyboardInterrupt: 

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.applications import ResNet50
import json
import os
import datetime

# ============ 1. إعداد المسارات ============ #
DATA_DIR = r"D:\github lite\cv_projects\Identifying and locating flags from aerial photography\data\flag_classification_lite"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ============ 2. Data Augmentation ============ #
def preprocess_function(img):
    img = tf.image.convert_image_dtype(img, tf.float32)
    img += 0.05 * tf.random.normal(tf.shape(img))
    return tf.clip_by_value(img, 0.0, 1.0)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_function,
    rotation_range=45,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    shear_range=0.15,
    brightness_range=(0.8, 1.2),
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = train_gen.num_classes
print(f"✅ Detected {NUM_CLASSES} classes.")

# ============ 3. بناء الموديل ============ #
base_model = ResNet50(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(*IMG_SIZE, 3)),
    layers.GaussianNoise(0.1),
    base_model,
    layers.BatchNormalization(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# ============ 4. Compile ============ #
initial_lr = 1e-2
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate=initial_lr,
    first_decay_steps=1000,
    t_mul=2.0,
    m_mul=0.9,
    alpha=1e-4
)

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=lr_schedule,
    weight_decay=1e-4
)

model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ============ 5. Callbacks ============ #
log_dir = os.path.join("logs", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs("models", exist_ok=True)

cb_list = [
    callbacks.ModelCheckpoint("models/best_model.h5", monitor='val_accuracy', save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.CSVLogger("training_log.csv"),
    callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
]

# ============ 6. التدريب ============ #
epochs = 50
model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=cb_list)

# ============ 7. Fine-tuning ============ #
base_model.trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=cb_list)

# ============ 8. حفظ النموذج ============ #
model.save("cnn_flag_classifier_resnet50.h5")
with open("class_labels.json", "w") as f:
    json.dump(train_gen.class_indices, f)

print("✅ Model (ResNet50) and labels saved.")


Found 2740 images belonging to 27 classes.
Found 673 images belonging to 27 classes.
✅ Detected 27 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 61s 1us/step


c:\Users\saher\anaconda3\envs\tf-gpu-lite\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 901ms/step - accuracy: 0.0537 - loss: 3.4868
Epoch 1: val_accuracy improved from -inf to 0.07875, saving model to models/best_model.h5


86/86 ━━━━━━━━━━━━━━━━━━━━ 109s 1s/step - accuracy: 0.0538 - loss: 3.4852 - val_accuracy: 0.0788 - val_loss: 3.2645 - learning_rate: 0.0098
Epoch 2/50
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 778ms/step - accuracy: 0.0773 - loss: 3.2139
Epoch 2: val_accuracy did not improve from 0.07875
86/86 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.0774 - loss: 3.2137 - val_accuracy: 0.0520 - val_loss: 3.2727 - learning_rate: 0.0093
Epoch 3/50
86/86 ━━━━━━━━━━━━━━━━━━━━ 0s 771ms/step - accuracy: 0.0710 - loss: 3.1836
Epoch 3: val_accuracy did not improve from 0.07875
86/86 ━━━━━━━━━━━━━━━━━━━━ 86s 998ms/step - accuracy: 0.0711 - loss: 3.1835 - val_accuracy: 0.0579 - val_loss: 3.1931 - learning_rate: 0.0084
Epoch 4/50
52/86 ━━━━━━━━━━━━━━━━━━━━ 25s 753ms/step - accuracy: 0.0638 - loss: 3.2266

KeyboardInterrupt: 